In [1]:
import pandas as pd
from scipy import stats

In [2]:
# Load cleaned property sales data
df = pd.read_csv("../data/processed/property_sales_clean.csv")

df.head()

,borough,neighborhood,building_class_category,tax_class_at_present,block,lot,building_class_at_present,address,zip_code,residential_units,...,tax_class_at_time_of_sale,building_class_at_time_of,sale_price,sale_date,apartment_number,price_per_sqft,sale_year,sale_month,sale_quarter,property_age
0,Manhattan,ALPHABET CITY,07 RENTALS - WALKUP APARTMENTS,2,376,41,C6,"745 EAST 6TH STREET, 1B",10009.0,28.0,...,2,C4,540000,2025-12-17,NaN,29.582557,2025,12,4,125.0
1,Manhattan,ALPHABET CITY,07 RENTALS - WALKUP APARTMENTS,2A,377,50,C2,"275 EAST 7TH STREET, 3",10009.0,5.0,...,2,C2,1250000,2026-05-27,NaN,198.601843,2026,5,2,126.0
2,Manhattan,ALPHABET CITY,07 RENTALS - WALKUP APARTMENTS,2,389,22,C7,208 EAST 7TH STREET,10009.0,28.0,...,2,C7,8600000,2026-03-09,NaN,442.888042,2026,3,1,126.0
3,Manhattan,ALPHABET CITY,07 RENTALS - WALKUP APARTMENTS,2A,390,60,C2,191 EAST 7 STREET,10009.0,5.0,...,2,C2,4665000,2026-03-25,NaN,1401.742788,2026,3,1,116.0
4,Manhattan,ALPHABET CITY,07 RENTALS - WALKUP APARTMENTS,2B,392,34,C7,153 AVENUE C,10009.0,8.0,...,2,C7,6850000,2026-05-14,NaN,1096.877502,2026,5,2,75.0


In [3]:
# Create sale price groups for each borough
manhattan = df[df["borough"] == "Manhattan"]["sale_price"]
brooklyn = df[df["borough"] == "Brooklyn"]["sale_price"]
queens = df[df["borough"] == "Queens"]["sale_price"]
bronx = df[df["borough"] == "Bronx"]["sale_price"]
staten_island = df[df["borough"] == "Staten Island"]["sale_price"]

In [ ]:
# Test whether average sale prices differ across boroughs with ANOVA
f_stat, p_value = stats.f_oneway(
    manhattan,
    brooklyn,
    queens,
    bronx,
    staten_island
)

print("F-statistic:", f_stat)
print("P-value:", p_value)

F-statistic: 332.9840013930241
P-value: 1.5449950196320902e-283


Finding: The ANOVA produced a p-value well below 0.05, indicating a statistically significant difference in average property sale prices across NYC boroughs.

In [5]:
# Keep rows with sale price and square footage
size_price = df[
    ["gross_square_feet", "sale_price"]
].dropna()

# Calculate correlation
correlation = size_price["gross_square_feet"].corr(
    size_price["sale_price"]
)

print("Correlation:", correlation)

Correlation: 0.3535516816279702


Finding: Gross square footage had a positive correlation of approximately 0.35 with sale price, indicating that larger properties tend to sell for higher prices, although the relationship is not particularly strong.

In [ ]:
# Compare average sale prices between Manhattan and Brooklyn with Two-Sample T-Test
t_stat, p_value = stats.ttest_ind(
    manhattan,
    brooklyn,
    equal_var=False
)

print("T-statistic:", t_stat)
print("P-value:", p_value)

T-statistic: 21.888723708046594
P-value: 1.0599240653782275e-104


Finding: The two-sample t-test produced a p-value well below 0.05, indicating a statistically significant difference in average property sale prices between Manhattan and Brooklyn.

In [7]:
# Create a contingency table of borough and property type
contingency_table = pd.crosstab(
    df["borough"],
    df["building_class_category"]
)

contingency_table

building_class_category,01 ONE FAMILY DWELLINGS,02 TWO FAMILY DWELLINGS,03 THREE FAMILY DWELLINGS,04 TAX CLASS 1 CONDOS,05 TAX CLASS 1 VACANT LAND,06 TAX CLASS 1 - OTHER,07 RENTALS - WALKUP APARTMENTS,08 RENTALS - ELEVATOR APARTMENTS,09 COOPS - WALKUP APARTMENTS,10 COOPS - ELEVATOR APARTMENTS,...,40 SELECTED GOVERNMENTAL FACILITIES,41 TAX CLASS 4 - OTHER,42 CONDO CULTURAL/MEDICAL/EDUCATIONAL/ETC,43 CONDO OFFICE BUILDINGS,44 CONDO PARKING,45 CONDO HOTELS,46 CONDO STORE BUILDINGS,47 CONDO NON-BUSINESS STORAGE,48 CONDO TERRACES/GARDENS/CABANAS,49 CONDO WAREHOUSES/FACTORY/INDUS
borough,,,,,,,,,,,,,,,,,,,,,
Bronx,819,904,352,73,93,19,260,63,78,735,...,0,21,2,1,10,0,2,0,0,0
Brooklyn,1830,2647,894,391,114,18,677,107,415,1792,...,0,30,3,35,489,0,32,126,10,12
Manhattan,120,78,37,15,0,1,379,139,675,5379,...,1,1,5,138,49,353,160,147,1,4
Queens,4507,2661,559,197,131,42,237,41,795,2677,...,0,19,8,74,509,0,39,58,7,1
Staten Island,2305,880,34,293,164,15,31,3,78,52,...,0,2,0,4,5,0,5,0,0,0


In [8]:
# Test the relationship between borough and property type
chi2, p_value, dof, expected = stats.chi2_contingency(
    contingency_table
)

print("Chi-square statistic:", chi2)
print("P-value:", p_value)

Chi-square statistic: 26622.092428189266
P-value: 0.0


Finding: The chi-square test produced a p-value well below 0.05, indicating a statistically significant association between borough and property type.